# HARs in Neurodegenerative Disease Risk - GP2 Project

Version 2 - Todos los HARs juntos

## Description

1. Description

- Getting Started
- Loading Python libraries
- Defining functions 
- Set Paths 
- Make working directory 

2. Installing packages

3. Copy over data

4. Create a covariate file with GP2 data

5. Annotation of the gene

6. Case/Control Frequencies

5. Calculate frequency in cases versus controls

6. Calculate frequency (homozygotes) in cases versus controls

7. Save out results

## Loading Python libraries

In [1]:
# Use the os package to interact with the environment
import os

#Import Sys
import sys as sys

# Use pathlib for file path manipulation
import pathlib
from pathlib import Path

# Parallel processing
from itertools import product
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# Import subprocess for easier management of bash commands
import subprocess

# Bring in Pandas for Dataframe functionality
import pandas as pd

# Numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
import seaborn as sns

# Import date and time for timestamps
from datetime import datetime, date
import time
import csv
import threading


print("Date of first run: 06-15-2025")
print("Date of last edits: 07-30-2025")


### Last iteration
print("Last iteration at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

### Package versions

print(f'''
=========================
    PACKAGE VERSIONS
=========================
pandas: {pd.__version__}
numpy: {np.__version__}
matplotlib: {matplotlib.__version__}
seaborn: {sns.__version__}
''')


Date of first run: 06-15-2025
Date of last edits: 07-30-2025
Last iteration at: 2026-04-16 16:28:18

    PACKAGE VERSIONS
pandas: 2.3.2
numpy: 2.2.6
matplotlib: 3.10.1
seaborn: 0.13.2



## Set paths

In [2]:
dataset = ["NBA", "WGS"]

### Release paths

In [3]:
## GP2 v11

release = 11

RELEASE_PATH = pathlib.Path(pathlib.Path.home(), f'workspace/gp2_tier2_eu_release{release}')
!ls -hal {RELEASE_PATH}

total 21K
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 clinical_data
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 clinical_exomes
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 imputed_genotypes
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 meta_data
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 raw_genotypes
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 raw_genotypes_flipped
-r--r--r--. 1 jupyter users 21K Dec  2 04:24 README_R11.txt
dr-xr-xr-x. 1 jupyter users   0 Apr 16 14:52 wgs


In [4]:
# GP2 R11

PATH_CLINICAL =           pathlib.Path(RELEASE_PATH, 'clinical_data/master_key_release11_final_vwb.csv')
PATH_EXTENDED_CLINICAL =  pathlib.Path(RELEASE_PATH, f'clinical_data/r{dataset}_extended_clinical_data_vwb.csv')

#NBA
PATH_NBA_GENO =           pathlib.Path(RELEASE_PATH, 'imputed_genotypes')
PATH_NBA_RAW =            pathlib.Path(RELEASE_PATH, 'raw_genotypes')
PATH_NBA_RELATED =        pathlib.Path(RELEASE_PATH, 'meta_data/related_samples')

#WGS
PATH_WGS_GENO =           pathlib.Path(RELEASE_PATH, 'wgs/deepvariant_joint_calling/plink')
PATH_WGS_RAW =            pathlib.Path(RELEASE_PATH, 'raw_genotypes')
PATH_WGS_RELATED =        pathlib.Path(RELEASE_PATH, 'wgs/deepvariant_joint_calling/related_samples')

## Workspace paths

In [5]:
### Define ANCESTRIES as a python list
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

In [6]:
## Get main directories

DIR_TOOL =  "/home/jupyter/tools"
DIR_HOME =  "/home/jupyter/workspace"
DIR_WSPS =  "/home/jupyter/workspace/ws_files"

DIR_ORIG =  "/home/jupyter/workspace/ws_files/Original_Files"

DIR_WORK_WGS =  "/home/jupyter/workspace/ws_files/Working_WGS"
DIR_WORK_NBA =  "/home/jupyter/workspace/ws_files/Working_NBA"

DIR_RESU_WGS =  "/home/jupyter/workspace/ws_files/Results_WGS"
DIR_RESU_NBA =  "/home/jupyter/workspace/ws_files/Results_NBA"

dirs = [DIR_TOOL, DIR_HOME, DIR_WSPS, DIR_ORIG, DIR_WORK_WGS, DIR_WORK_NBA, DIR_RESU_WGS, DIR_RESU_NBA]

## Subdirs for analyses
subdirs = ["InputFiles", "VariantDescriptives", "Association", "GLM", "Burden"]

for dir in dirs:
    os.makedirs(dir, exist_ok=True)

for ancestry in ANCESTRIES:
    for subdir in subdirs:
        os.makedirs(pathlib.Path(DIR_WORK_WGS, ancestry, subdir), exist_ok=True)
        os.makedirs(pathlib.Path(DIR_WORK_NBA, ancestry, subdir), exist_ok=True)    


In [7]:
# Extract HARs from tsv file and create a dictionary with HAR name as key and chrom, start, end as values

with open(f'{DIR_ORIG}/HAR_list_phase_1.tsv', 'r') as f:
    reader = csv.reader(f, delimiter='\t')
    HARS_DICT = {
        row[3]: {
            'name':  row[3],
            'chrom': row[0].replace('chr', ''),
            'start': int(row[1]),
            'end':   int(row[2]),
        }
        for row in reader
}

In [8]:
# print(HARS_DICT)
# for HAR in HARS_DICT:
#    print(f"{HAR}")

## Install Packages

In [9]:
%%bash
%%capture

## Check if plink1.9 is installed
if test -e /home/jupyter/tools/plink1.9; then
    echo "Plink is already installed in /home/jupyter/tools"
else
    echo "Plink is not installed"

    cd /home/jupyter/tools

    wget -nc http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

## ------------------------------## 
## Check if plink2.0 is installed
if test -e /home/jupyter/tools/plink2; then
    echo "Plink2 is already installed in /home/jupyter/tools"
else
    echo "Plink2 is not installed"
    cd /home/jupyter/tools
    
    wget -nc http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

    unzip -o plink2_linux_x86_64_latest.zip
fi

chmod u+x /home/jupyter/tools/plink1.9
chmod u+x /home/jupyter/tools/plink2

bash: line 1: fg: no job control


Plink is not installed


--2026-04-15 19:34:40--  http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 54.231.160.112, 54.231.169.160, 52.217.229.184, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|54.231.160.112|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8708135 (8.3M) [application/zip]
Saving to: ‘plink_linux_x86_64_20190304.zip’

     0K .......... .......... .......... .......... ..........  0%  297K 28s
    50K .......... .......... .......... .......... ..........  1%  595K 21s
   100K .......... .......... .......... .......... ..........  1%  594K 19s
   150K .......... .......... .......... .......... ..........  2%  269M 14s
   200K .......... .......... .......... .......... ..........  2%  595K 14s
   250K .......... .......... .......... .......... ..........  3%  347M 12s
   300K .......... .......... .......... .......... ..........  4%  405M 10s
   350K .......... .......... .......... .......

Archive:  plink_linux_x86_64_20190304.zip
  inflating: plink                   
  inflating: LICENSE                 
  inflating: toy.ped                 
  inflating: toy.map                 
  inflating: prettify                
Plink2 is not installed


--2026-04-15 19:34:41--  http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip
Resolving s3.amazonaws.com (s3.amazonaws.com)... 54.231.160.112, 54.231.169.160, 52.217.229.184, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|54.231.160.112|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7604901 (7.3M) [application/zip]
Saving to: ‘plink2_linux_x86_64_latest.zip’

     0K .......... .......... .......... .......... ..........  0%  297K 25s
    50K .......... .......... .......... .......... ..........  1%  594K 19s
   100K .......... .......... .......... .......... ..........  2%  133M 12s
   150K .......... .......... .......... .......... ..........  2%  175M 9s
   200K .......... .......... .......... .......... ..........  3%  596K 10s
   250K .......... .......... .......... .......... ..........  4%  302M 8s
   300K .......... .......... .......... .......... ..........  4%  415M 7s
   350K .......... .......... .......... .......... .

Archive:  plink2_linux_x86_64_latest.zip
  inflating: plink2                  
  inflating: vcf_subset              
  inflating: intel-simplified-software-license.txt  


In [10]:
%%capture
%%bash

# Install ANNOVAR: We are adding the download link after registration on the annovar website
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then
    echo "annovar is already installed in /home/jupyter/tools/notebooks"

else
    echo "annovar is not installed"
    cd /home/jupyter/tools
    
    wget -nc http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz
    
    tar -xvzf annovar.latest.tar.gz
fi

In [ ]:
%%bash
# %%capture

cd /home/jupyter/tools/annovar/

# Install ANNOVAR: Download resources for annotation

# perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/ #latest version of dbNSFP
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20250721 humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_exome humandb/
perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar refGene humandb

NOTICE: Web-based checking to see whether ANNOVAR new version is available ... Done
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_dbnsfp47a.txt.gz ... OK
NOTICE: Downloading annotation database http://www.openbioinformatics.org/annovar/download/hg38_dbnsfp47a.txt.idx.gz ... OK
NOTICE: Uncompressing downloaded files


In [ ]:
%%capture
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then
    echo "RVtests is already installed in /home/jupyter/tools/"
else
    echo "RVtests is not installed"
    
    mkdir /home/jupyter/tools/rvtests
    cd /home/jupyter/tools/rvtests
    
    wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 
    
    tar -zxvf rvtests_linux64.tar.gz
fi

chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

In [ ]:
%%bash
%%capture

! wget -O {DIR_TOOL}/refFlat_HG38_all_chr.txt.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz 
! gunzip -f {DIR_TOOL}/refFlat_HG38_all_chr.txt.gz

cd /home/jupyter/tools/
rm *.zip
rm *.gz
rm toy*
rm prettify
rm vcf_subset
rm LICENSE
rm ./rvtests/*.gz
rm -r  ./rvtests/exampl

## Create a covariate file with GP2 data

In [19]:
# Let's load the master key
key1 = pd.read_csv(PATH_CLINICAL, low_memory=False)
print(f'Clinical data (num rows, num columns): {key1.shape}')
pd.set_option('display.max_columns', None)
key1.head()

Clinical data (num rows, num columns): (130274, 32)


,study,FID,GP2ID,nba,wgs,clinical_exome,extended_clinical_data,nba_prune_reason,nba_related,nba_label,wgs_prune_reason,wgs_label,study_arm,study_type,diagnosis,baseline_GP2_phenotype_for_qc,baseline_GP2_phenotype,biological_sex_for_qc,age_at_sample_collection,age_of_onset,age_at_diagnosis,age_at_death,age_at_last_follow_up,race_for_qc,family_history_for_qc,region_for_qc,manifest_id,genotyping_site,sample_type,amppd_wgs,GDPR,flag
0,24HR,24HR_000001,24HR_000001,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Female,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
1,24HR,24HR_000002,24HR_000002,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
2,24HR,24HR_000003,24HR_000003,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
3,24HR,24HR_000004,24HR_000004,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
4,24HR,24HR_000005,24HR_000005,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN


In [20]:
# Subsetting to keep only a few columns 
key = key1[['GP2ID', 'GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label','wgs_label']]

# 'nba_GP2sampleID_r11'
# Renaming the columns

key.columns = ['IID', 'IID1_OLD', 'phenotype', 'SEX', 'AGE', 'AAO', 'nba_label', 'wgs_label']
key

,IID,IID1_OLD,phenotype,SEX,AGE,AAO,nba_label,wgs_label
0,24HR_000001,24HR_000001,PD,Female,NaN,NaN,EUR,EUR
1,24HR_000002,24HR_000002,PD,Male,NaN,NaN,EUR,EUR
2,24HR_000003,24HR_000003,PD,Male,NaN,NaN,EUR,EUR
3,24HR_000004,24HR_000004,PD,Male,NaN,NaN,EUR,EUR
4,24HR_000005,24HR_000005,PD,Male,NaN,NaN,EUR,EUR
...,...,...,...,...,...,...,...,...
130269,YMS_000080,YMS_000080,Other,Male,66.0,NaN,AJ,NaN
130270,YMS_000081,YMS_000081,Other,Male,71.0,NaN,AAC,NaN
130271,YMS_000082,YMS_000082,PD,Male,76.0,71.0,AJ,NaN
130272,YMS_000083,YMS_000083,Other,Male,69.0,NaN,AJ,NaN


In [21]:
key["label"] = key["nba_label"].combine_first(key["wgs_label"])
key = key.drop(columns=["nba_label", "wgs_label"])
key

/tmp/ipykernel_403/428804967.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  key["label"] = key["nba_label"].combine_first(key["wgs_label"])


,IID,IID1_OLD,phenotype,SEX,AGE,AAO,label
0,24HR_000001,24HR_000001,PD,Female,NaN,NaN,EUR
1,24HR_000002,24HR_000002,PD,Male,NaN,NaN,EUR
2,24HR_000003,24HR_000003,PD,Male,NaN,NaN,EUR
3,24HR_000004,24HR_000004,PD,Male,NaN,NaN,EUR
4,24HR_000005,24HR_000005,PD,Male,NaN,NaN,EUR
...,...,...,...,...,...,...,...
130269,YMS_000080,YMS_000080,Other,Male,66.0,NaN,AJ
130270,YMS_000081,YMS_000081,Other,Male,71.0,NaN,AAC
130271,YMS_000082,YMS_000082,PD,Male,76.0,71.0,AJ
130272,YMS_000083,YMS_000083,Other,Male,69.0,NaN,AJ


In [47]:
PATH_NBA_RELATED = pathlib.Path(RELEASE_PATH, 'meta_data/related_samples')
print(f'{PATH_NBA_RELATED}')
print(f'{{PATH_{set}_RELATED}}')

print(f'{MAIN}')

/home/jupyter/workspace/gp2_tier2_eu_release11/meta_data/related_samples
{PATH_NBA_RELATED}
{DIR_WORK_NBA}/AAC


In [60]:
for data in dataset: 
    for ANCESTRY in ANCESTRIES:

        print(f'\n===== WORKING ON: {data} - {ANCESTRY} =====')

        DIR_WORK_TEMP =f"DIR_WORK_{data}"
        DIR_WORK = globals()[DIR_WORK_TEMP]               
        
        PATH_REL_TEMP = f"PATH_{data}_RELATED"
        PATH_REL = globals()[PATH_REL_TEMP]

        PATH_RAW_TEMP = f"PATH_{data}_RAW"
        PATH_RAW = globals()[PATH_RAW_TEMP]

        MAIN = f"{DIR_WORK}/{ANCESTRY}"
                
        ## Subset to keep ANCESTRY of interest 
        ANCESTRY_key = key[key['label']==ANCESTRY].copy()
        ANCESTRY_key.reset_index(drop=True)
            
        # Load information about related individuals in the ANCESTRY analyzed
        if data == "NBA":
            related_df = pd.read_csv(f'{PATH_REL}/{ANCESTRY}_release11_vwb.related')
        elif data == "WGS":
            try:
                related_df = pd.read_csv(f'{PATH_REL}/{ANCESTRY}/{ANCESTRY}_release11.related')
            except Exception as e:
                print(e)
                pass
            
        print(f'Related individuals: {related_df.shape}')
        
        # Make a list of just one set of related people
        related_list = list(related_df['IID1'])
        
        # Check value counts of related and remove only one related individual
        ANCESTRY_key = ANCESTRY_key[~ANCESTRY_key["IID1_OLD"].isin(related_list)]
        # Check size
        print(f'Unrelated individuals: {ANCESTRY_key.shape}')
            
        # Convert phenotype to binary (1/2)
        ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
        # PD = 2; control = 1
        pheno_mapping = {"PD": 2, "Control": 1}
        ANCESTRY_key['PHENO'] = ANCESTRY_key['phenotype'].map(pheno_mapping).astype('Int64')
    
        # Check value counts of pheno
        ANCESTRY_key['PHENO'].value_counts(dropna=False)
        
        ## Get the PCs
        pcs = pd.read_csv(f'{PATH_RAW}/{ANCESTRY}/{ANCESTRY}_release11_vwb.eigenvec', sep='\t')
        
        #Select just first 10 PCs
        selected_columns = ['FID', 'IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
        pcs = pd.DataFrame(data=pcs.iloc[:, 0:12].values, columns=selected_columns)
    
        # Drop the first row (since it's now the column names)
        pcs = pcs.drop(0)
    
        # Reset the index to remove any potential issues
        pcs = pcs.reset_index(drop=True)
        
        # Check size
        print(f'PCs: {pcs.shape}')
    
        # Check value counts of SEX
        sex_og_values = ANCESTRY_key['SEX'].value_counts(dropna=False)
        print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
        
        # Convert sex to binary (1/2)
        ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
        # Female = 2; Male = 1
        sex_mapping = {"Female": 2, "Male": 1}
        ANCESTRY_key['SEX'] = ANCESTRY_key['SEX'].map(sex_mapping).astype('Int64')
        
        # Check value counts of SEX after recoding
        sex_recode_values = ANCESTRY_key['SEX'].value_counts(dropna=False)
        print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
        
        ## Make covariate file
        df = pd.merge(pcs, ANCESTRY_key, on='IID', how='left')
        print(f'\nCheck columns for covariate file: {df.columns}')
        
        #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
        #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
        #Uppercase column name is ok
        #See https://zhanxw.github.io/rvtests/#phenotype-file
        df['FATID'] = 0
        df['MATID'] = 0
    
        ## Clean up and keep columns we need 
        final_df = df[['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']].copy()
        final_df.columns = ['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
    
        print(f'\nCheck columns for Final covariate file: {final_df.columns}')
    
        ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
        #Leave missing values as NA
        
        #Check number of PD cases missing age
        pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
        print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
        
        #Check number of controls missing age
        control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
        print(f'Number of controls missing age: {control_missAge.shape[0]}')
    
        ## Make file of sample IDs to keep 
        samples_toKeep = final_df[['FID', 'IID']].copy()
        
        samplestokeep_path = pathlib.Path(pathlib.Path.home(), f'{MAIN}/InputFiles/{ANCESTRY}.samplestokeep')
        
        # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
        os.makedirs(samplestokeep_path.parent, exist_ok=True)
        print(f'Created {samplestokeep_path.parent}')
        
        samples_toKeep.to_csv(samplestokeep_path, sep = '\t', index=False, header=None)
    
        finaldf_path = pathlib.Path(pathlib.Path.home(), f'{MAIN}/InputFiles/{ANCESTRY}_covariate_file.txt')
        
        # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
        os.makedirs(finaldf_path.parent, exist_ok=True)
        print(f'Created {finaldf_path.parent}')
        
        final_df.to_csv(finaldf_path, sep = '\t', na_rep='NA', index=False)


===== WORKING ON: NBA - AAC =====
Related individuals: (12, 9)
Unrelated individuals: (1584, 7)
PCs: (1381, 12)
Sex value counts - original:
 SEX
Female                        861
Male                          721
Other/Unknown/Not Reported      2
Sex value counts - recoded:
SEX
2       861
1       721
<NA>      2

Check columns for covariate file: Index(['FID', 'IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8',
       'PC9', 'PC10', 'IID1_OLD', 'phenotype', 'SEX', 'AGE', 'AAO', 'label',
       'PHENO'],
      dtype='object')

Check columns for Final covariate file: Index(['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO', 'PC1', 'PC2',
       'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10'],
      dtype='object')
Number of PD cases missing age: 82
Number of controls missing age: 22
Created /home/jupyter/workspace/ws_files/Working_NBA/AAC/InputFiles
Created /home/jupyter/workspace/ws_files/Working_NBA/AAC/InputFiles

===== WORKING ON: NBA - AFR =====
Related in

## Annotation of the gene

### Extract the region using PLINK in hg38
Region extractor was compiled independantly to improve processing in High CPU. Run as follows:

**nohup python3 regionExtractor.py > run.log 2>&1 & echo $! > run.pid**

Will extract every HAR into: *MAIN/ANCESTRY/InputFiles/Indiv_HARS*

Failed HARs in: *MAIN/ANCESTRY/failed_HARs_(NBA/WGS)_ANCESTRY.tsv*

**Useful commands In terminal:**

*Live output*:
+ tail -f run.log

*Count progress*:
+ grep -c "DONE" run.log
+ grep -c "SKIP" run.log
+ grep -c "ERROR" run.log

*Check if still running*:
+ cat run.pid | xargs kill -0 && echo "Running" || echo "Finished"

*Kill process*:
+ kill -9 $(cat run.pid)

## Annotate using ANNOVAR

In [ ]:
def regionAnnotator(set, HAR, ANCESTRY):
    print(f"Extracting {HAR} from {ANCESTRY} {set} Release 11 data.")

    MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_WORK"
    
    input_vcf = f"{MAIN}/{ANCESTRY}/InputFiles/{ANCESTRY}_{HAR}.vcf.gz"
    output_base = f"{MAIN}/{ANCESTRY}/InputFiles/{ANCESTRY}_{HAR}.annovar"
    
    cmd = [
        "perl", f"{DIR_TOOL}/annovar/table_annovar.pl",
        input_vcf,
        f"{DIR_TOOL}/annovar/humandb/",
        "-buildver", "hg38",
        "-out", output_base,
        "-remove", 
        "-protocol", "refGene,clinvar_20250721,dbnsfp47a,gnomad41_genome,gnomad41_exome", 
        "-operation", "g,f,f,f,f", 
        "--nopolish",
        "-nastring", ".",
        "-vcfinput"
    ]
    
    try:
        subprocess.run(cmd, check=True)
        print(f"\nSUCCESS: {ANCESTRY}, {HAR}\n")
    except subprocess.CalledProcessError as e:
        print(f"\nERROR in {ANCESTRY}, {HAR}: {e}\n")
    
    return f"{output_base}.hg38_multianno.txt" # Return the path so you can use it

# Logic for workers and product
def regionAnnotator_wrapper(args):
    return regionAnnotator(*args)

# Explicitly use .keys() if GENE_DICT is a dictionary
args = list(product(HARS_DICT.keys(), ANCESTRIES, dataset)) 

max_workers = max(1, len(ANCESTRIES) // 2)

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    # results will now be a list of paths to your output files
    results = list(executor.map(regionAnnotator_wrapper, args))

In [ ]:
for HAR in HARS_DICT:
    for set in dataset:
        for ANCESTRY in ANCESTRIES:
            MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_WORK"

            anno_dir = f"{MAIN}/{ANCESTRY}/InputFiles"

            print(f"Working on {dataset} - {ANCESTRY}")
            
            df = pd.read_csv(f'{anno_dir}/{ANCESTRY}_{HAR}.annovar.hg38_multianno.txt', sep='\t')
            subset = df[["Chr", "Start", "End", "Ref", "Alt", "Func.refGene", "Gene.refGene", "ExonicFunc.refGene", "CLNSIG", "REVEL_score", "REVEL_rankscore", "CADD_raw", "CADD_raw_rankscore", "CADD_phred", "gnomad41_genome_fafmax_faf95_max"]]
            
            subset.to_csv(f'{anno_dir}/{ANCESTRY}_{HAR}.annovar.hg38_multianno_cropped.txt', sep='\t', index=False)
            
            print(f"{dataset} - {ANCESTRY} finished")

In [ ]:
for HAR in HARS_DICT:
    for set in dataset:
        for ANCESTRY in ANCESTRIES:
            print(f'WORKING ON: {ANCESTRY}')
            BUILD = "hg38"

            MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_WORK"
            
            inputPrefix = f"{MAIN}/{ANCESTRY}/InputFiles/{ANCESTRY}_{HAR}"
            # Read in ANNOVAR multianno file
            gene = pd.read_csv(f'{inputPrefix}.annovar.{BUILD}_multianno_cropped.txt', sep = '\t', header = 0)
                
            #Filter for the correct gene name (sometimes other genes are also included)
            gene['gnomad41_genome_fafmax_faf95_max'] = pd.to_numeric(gene['gnomad41_genome_fafmax_faf95_max'], errors='coerce')
            gene['CADD_phred'] = pd.to_numeric(gene['CADD_phred'], errors='coerce')
            gene['REVEL_score'] = pd.to_numeric(gene['REVEL_score'], errors='coerce')
            
                
            #Print number of variants in the different categories
            results = [] 
            
            intronic = gene[gene['Func.refGene']== 'intronic']
            upstream = gene[gene['Func.refGene']== 'upstream']
            downstream = gene[gene['Func.refGene']== 'downstream']
            utr5 = gene[gene['Func.refGene']== 'UTR5']
            utr3 = gene[gene['Func.refGene']== 'UTR3']
            splicing = gene[gene['Func.refGene']== 'splicing']
            exonic = gene[gene['Func.refGene']== 'exonic']
            stopgain = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'stopgain')]
            stoploss = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'stoploss')]
            startloss = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'startloss')]
            frameshift_deletion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'frameshift deletion')]
            frameshift_insertion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'frameshift insertion')]
            nonframeshift_deletion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonframeshift deletion')]
            nonframeshift_insertion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonframeshift insertion')]
            coding_nonsynonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonsynonymous SNV')]
            coding_synonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'synonymous SNV')]
                
            #Predictors
            maf_01 = gene[gene['gnomad41_genome_fafmax_faf95_max']<0.01]
            maf_03 = gene[gene['gnomad41_genome_fafmax_faf95_max']<0.03]
            CADD_20 = gene[gene['CADD_phred']>20]
            REVEL_50 = gene[gene['REVEL_score']>0.5]
            lof = gene[(gene['Func.refGene'] == 'splicing') | (gene['ExonicFunc.refGene'] == 'stopgain') | (gene['ExonicFunc.refGene'] == 'stoploss') | (gene['ExonicFunc.refGene'] == 'startloss') | (gene['ExonicFunc.refGene'] == 'frameshift deletion') | (gene['ExonicFunc.refGene'] == 'frameshift insertion')]
            
            #CLNSIG annotation
            pathogenic =    gene[(gene['CLNSIG'] == 'Pathogenic')]
            likely_patho =  gene[(gene['CLNSIG'] == 'Likely_pathogenic')]
            var_unk_sig =   gene[(gene['CLNSIG'] == 'Uncertain_significance')]
            likely_benign = gene[(gene['CLNSIG'] == 'Likely_benign')]
            benign =        gene[(gene['CLNSIG'] == 'Benign')]
        
            variantsDict = {
                    "Total variants": gene,
                    "Intronic": intronic,
                    "Upstream": upstream,
                    "Downstream": downstream,
                    "UTR3": utr3,
                    "UTR5": utr5,
                    "Splicing": splicing,
                    "Exonic": exonic,
                    "Stopgain": stopgain,
                    "Stoploss": stoploss,
                    "Frameshift deletion": frameshift_deletion,
                    "Frameshift insertion": frameshift_insertion,
                    "Non-frameshift insertion": nonframeshift_insertion,
                    "Non-frameshift deletion": nonframeshift_deletion,
                    "Synonymous": coding_synonymous,
                    "Nonsynonymous": coding_nonsynonymous,
                    "LOF": lof,
                    "MAF01": maf_01,
                    "MAF03": maf_03,
                    "REVEL>0.5": REVEL_50,
                    "CADD>20": CADD_20,
                    "Pathogenic": pathogenic,
                    "Likely Pathogenic": likely_patho,
                    "Uncertain significance": var_unk_sig,
                    "Likely Benign": likely_benign,
                    "Benign": benign,
                }

                        
            variantSummary = pd.DataFrame({
                    "variantType": list(variantsDict.keys()),
                    "count": [len(variant) for variant in variantsDict.values()]
                })
            
            ## Save numbers of variants for future reference
            os.makedirs(f"{MAIN}/{ANCESTRY}/VariantDescriptives", exist_ok = True)
            variantSummary.to_csv(f"{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_variantCounts_refGene.csv", index=False)
        
            ## Print variants which are not "0"
            allvar = variantSummary
            allVariantSummary = variantSummary.to_string(index=False, header=False)
            filtered = variantSummary[variantSummary['count'] > 0]
            filteredVariantSummary = filtered.to_string(index=False, header=False)
            variantSummary_T = variantSummary.T.reset_index()
            variantSummary_T.columns = variantSummary_T.iloc[0]
            variantSummary_T = variantSummary_T[1:].reset_index(drop=True)
            descriptives = variantSummary_T.copy()
            descriptives.insert(0, 'Ancestry', ANCESTRY)
            descriptives.insert(1, 'Gene', HAR)
            descriptives.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_variant_descriptives', sep="\t", index=False)
            filteredVariantSummary = filtered.to_string(index=False, header=False)                
                
            print(f"""
                =====================================
                    VARIANT SUMMARY FOR {ANCESTRY}
                    -----------------------
                    {variantSummary} 
                =====================================
                    """)
                
            ## For rvtests
            ## Uncomment to_csv commands if you need to debug or full outputs
            all_variants = gene
            all_variants.to_csv(f"{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_all_variants.csv", index=False)

            # Potential functional: These are variants annotated as frameshift, nonframeshift, startloss, stoploss, stopgain, splicing, missense, exonic, UTR5, UTR3, upstream (-100bp), downstream (+100bp), or ncRNA.
            potentially_functional = gene[gene['Func.refGene'] != 'intronic']
            potentially_functional.to_csv(f"{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_variants_full_potentially_functional.csv", index=False)
                    
            # Coding: These are variants annotated as frameshift, nonframeshift, startloss, stoploss, stopgain, splicing, or missense.
            coding_variants = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] != 'synonymous SNV')]
            coding_variants.to_csv(f"{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_variants_full_coding.csv", index=False)

            potencially_pathogenic = gene[(gene['CLNSIG'] == 'Pathogenic') | (gene['CLNSIG'] == 'Likely_pathogenic') | (gene['CADD_phred']>20) | (gene['REVEL_score']>0.5)]
            potencially_pathogenic.to_csv(f"{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_variants_potencially_pathogenic.csv", index=False)

            # Save in PLINK format
            variants_toKeep = potentially_functional[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}.potentially_functional.variantstoKeep.txt', sep="\t", index=False, header=False)
                
            variants_toKeep2 = coding_variants[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep2.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}.coding_variants.variantstoKeep.txt', sep="\t", index=False, header=False)
                
            variants_toKeep4 = coding_nonsynonymous[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep4.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR }.exonic.variantstoKeep.txt', sep="\t", index=False, header=False)
                    
            variants_toKeep5 = all_variants[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep5.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_all_variants.variantstoKeep.txt', sep="\t", index=False, header=False)

            variants_toKeep6 = maf_01[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep6.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_maf01.variantstoKeep.txt', sep="\t", index=False, header=False)

            variants_toKeep7 = maf_03[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep7.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_maf03.variantstoKeep.txt', sep="\t", index=False, header=False)

            variants_toKeep8 = lof[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep8.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_lof.variantstoKeep.txt', sep="\t", index=False, header=False)

            variants_toKeep9 = potencially_pathogenic[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
            variants_toKeep9.to_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_potencially_pathogenic.variantstoKeep.txt', sep="\t", index=False, header=False)


In [ ]:
for HAR in HARS_DICT:
    for set in dataset:

        all_variant_stats = []
        patho_variant_stats = []

        for ANCESTRY in ANCESTRIES:
            MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_WORK"
            descriptives = pd.read_csv(f'{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}_variant_descriptives', sep="\t", header=0)
            all_variant_stats.append(descriptives)
            
        all_variant_stats = pd.concat(all_variant_stats)
        all_variant_stats.to_csv(f'f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_RESU/HAR_{HAR}_{analysis}_AllVariantsStats.csv', index=False)

## Burden Analyses using RVTests

In [ ]:
#Burden tests are performed in three groups: maf < 0.01, maf < 0.03 and potencially pathogenic.

burden_class = ['maf01', 'maf03', 'potencially_pathogenic']
for i in burden_class:  
    print(i)

In [ ]:
## Extract variants of interest as vcf

def burdenPrep(HAR, set, ANCESTRY, burden_class):

    MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_WORK"

    ## Input files
    if set == "NBA":    
        inputRawPFILE = f"{PATH_NBA_IMPUTED}/{ANCESTRY}/chr7_{ANCESTRY}_release11_vwb"
    elif set == "WGS":
        inputRawPFILE = f"{PATH_WGS_GENO}/{ANCESTRY}/chr7_{ANCESTRY}_release11_vwb"

    samplesToKeep = f"{MAIN}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    variantsToKeep = f"{MAIN}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{HAR}.{burden_class}.variantstoKeep.txt"
    outputVCF = f"{MAIN}/{ANCESTRY}/Burden/{ANCESTRY}_{HAR}.{burden_class}"
    
    ## Define plink2 command for extraction
    variantExtractor = [
        f"{DIR_TOOL}/plink2",
        "--pfile", inputRawPFILE,
        "--keep", samplesToKeep,
        "--extract", "range", variantsToKeep,
        "--recode", "vcf-iid",
        "--out", outputVCF
    ]

        ## Run plink2 command as bash subprocess
    subprocess.run(variantExtractor, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    vcfGzip = [
        "bgzip",
        "-f", f"{outputVCF}.vcf"
        ]

        ## Run plink2 command as bash subprocess
    subprocess.run(vcfGzip, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    vcfIndex = [
        "tabix",
        "-f", "-p", "vcf", f"{outputVCF}.vcf.gz"      
        ]    
    ## Run plink2 command as bash subprocess
    subprocess.run(vcfIndex, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

In [ ]:
for HAR in HARS_DICT:
    for set in dataset:
        for ancestry in ANCESTRIES:
            print(f'WORKING ON: HAR:{HAR} {set} {ancestry}\n')
            for burden in burden_class: 
                print(f'\tClass: {burden}')
                try:
                    burdenPrep(HAR, set, ancestry, burden)
                    print(f'\tHAR:{HAR} {set} {ancestry}, {burden} - DONE\n')
                except Exception as e:
                    print(f'HAR:{HAR} {set} {ancestry}, {burden} - FAIL\n')

In [ ]:
## Run burden test with RVtests

def burdenRVTests(HAR, set, ANCESTRY, burden_class):
   
    MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}_WORK"

    ## Get ANCESTRY name
    inputVCF = f"{MAIN}/{ANCESTRY}/Burden/{ANCESTRY}_{HAR}.{burden_class}.vcf.gz"
    outputBURDEN = f"{MAIN}/{ANCESTRY}/Burden/{ANCESTRY}_{HAR}_{burden_class}.burden"
    covariate = f"{MAIN}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"
    refFlat = f"{DIR_TOOL}/refFlat_HG38_all_chr.txt"
    
    burdenRVTests = [
        #RVtests with covariates 
        #Make sure the pheno and covariate file starts with the first 5 columsn: fid, iid, fatid, matid, sex
        #The pheno-name flag only works when the pheno/covar file is structured properly
        f"{DIR_TOOL}/rvtests/executable/rvtest", 
        "--noweb", 
        "--hide-covar",
        "--out", outputBURDEN,
        "--kernel", "skat,skato",
        "--inVcf", inputVCF,
        "--pheno", covariate,
        "--pheno-name", "PHENO",
        "--gene", HAR,
        "--geneFile", refFlat,
        "--covar", covariate,
        "--covar-name", "SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
    ]
    
    ## Run plink2 command as bash subprocess
    subprocess.run(burdenRVTests, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

In [ ]:
refFlat = f"{DIR_TOOL}/refFlat_HG38_all_chr.txt"
for HAR in HARS_DICT:
    for set in dataset:
        for ancestry in ANCESTRIES:
            print(f'WORKING ON: HAR:{HAR} {set} {ancestry}\n')
            for burden in burden_class: 
                print(f'\tClass: {burden}')
                try:
                    burdenRVTests(HAR, set, ancestry, burden)
                    print(f'\t{HAR} {set} {ancestry}, {burden} - DONE\n')
                except Exception as e:
                    print(f'{HAR} {set} {ancestry}, {burden} - FAIL: {e}\n')

In [ ]:
for HAR in HARS_DICT:
    for set in dataset:

        skat_df = []
        skatO_df = []

        for ANCESTRY in ANCESTRIES:
            for burden in burden_class:

                MAIN = f"{DIR_WORK}/HAR_{HAR}/{set}_{HAR}"
            
                path_skat = Path(f"{MAIN}_WORK/{ANCESTRY}/Burden/{ANCESTRY}_{HAR}_{burden}.burden.Skat.assoc")
                path_skatO = Path(f"{MAIN}_WORK/{ANCESTRY}/Burden/{ANCESTRY}_{HAR}_{burden}.burden.SkatO.assoc")            
                    
                try:
                    if path_skat.is_file():
                        df = pd.read_csv(path_skat, sep="\t")
                        df.insert(0, "ANCESTRY", ANCESTRY)
                        df.insert(1, "GENE", HAR)
                        df.insert(2, "CLASS", burden)
                        skat_df.append(df)
                except:
                    pass
                
                try:
                    if path_skatO.is_file():
                        df = pd.read_csv(path_skatO, sep="\t")
                        df.insert(0, "ANCESTRY", ANCESTRY)
                        df.insert(1, "GENE", HAR)
                        df.insert(2, "CLASS", burden)
                        skatO_df.append(df)
                except:
                    pass

        skat_df = pd.concat(skat_df, ignore_index=True) if skat_df else pd.DataFrame()
        skatO_df = pd.concat(skatO_df, ignore_index=True) if skatO_df else pd.DataFrame()

        skat_df
        skatO_df

        skat_df.to_csv(f"{MAIN}_RESU/HAR_{HAR}._{set}_BURDEN.SKAT.tsv", sep='\t', index=False)
        skatO_df.to_csv(f"{MAIN}_RESU/HAR_{HAR}._{set}_BURDEN.SKAT-O.tsv", sep='\t', index=False)

## Case/Control Analysis

### Glossary

- CHR Chromosome code
- SNP Variant identifier
- A1 Allele 1 (usually minor)
- A2 Allele 2 (usually major)
- MAF Allele 1 frequency in all subjects
- F_A/MAF_A Allele 1 frequency in cases
- F_U/MAF_U Allele 1 frequency in controls
- NCHROBS_A Number of case allele observations
- NCHROBS_U Number of control allele observations

In [ ]:
# variant_classes = ['all_variants', 'potentially_functional', 'coding_variants','loss_of_function', 'exonic']
# print(variant_classes)

['all_variants', 'potentially_functional', 'coding_variants', 'loss_of_function', 'exonic']


In [ ]:
for anc in ANCESTRIES:
    print(f"Working on {anc}")
    
    inputpfile = f"{ANA_DIR}/{anc}/InputFiles/{anc}_CADPS2"
    outbfile = f"{ANA_DIR}/{anc}/InputFiles/{anc}_CADPS2"
    
    makebfiles = [
        f"{TOOL_DIR}/plink2",
          "--pfile", inputpfile,
          "--make-bed",
          "--out", outbfile
    ]
    
    subprocess.run(makebfiles, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

    print(f"{anc} done, BFILES saved in {ANA_DIR}/{anc}/InputFiles/{anc}_CADPS2")

Working on AAC
AAC done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/AAC/InputFiles/AAC_CADPS2
Working on AFR
AFR done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/AFR/InputFiles/AFR_CADPS2
Working on AJ
AJ done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/AJ/InputFiles/AJ_CADPS2
Working on AMR
AMR done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/AMR/InputFiles/AMR_CADPS2
Working on CAS
CAS done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/CAS/InputFiles/CAS_CADPS2
Working on EAS
EAS done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/EAS/InputFiles/EAS_CADPS2
Working on EUR
EUR done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/EUR/InputFiles/EUR_CADPS2
Working on MDE
MDE done, BFILES saved in /home/jupyter/workspace/ws_files/NahuTestR11/WGS/MDE/InputFiles/MDE_CADPS2
Working on SAS
SAS done, BFILES saved in /home/jupyter/workspace/ws_files/Na

In [ ]:
## extract region using plink

def variantAssoc(GENE_NAME, ANCESTRY, var_class=None):
    
    ## Get ANCESTRY name
    samplesToKeep = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    inputFile = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
    prefix = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}"
    covariate = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"

    recodeA = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputFile,
        "--keep", samplesToKeep,
        "--recode", "A",
        "--out", prefix
    ]
    
    subprocess.run(recodeA, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
    ## Define plink2 command for extraction
    plinkAssoc_base = [
        f"{TOOL_DIR}/plink1.9",
        "--bfile", inputFile,
        "--keep", samplesToKeep,
        "--assoc",
        "--pheno", covariate,
        "--mpheno", "5",
        "--max-maf", "0.05",
        "--mac", "2",
        "--adjust",
        "--allow-no-sex",
        "--ci", "0.95",
    ]

    ## Define plink2 command for extraction
    plinkGLM_base = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputFile,
        "--keep", samplesToKeep,
        "--glm",
        "hide-covar",
        "cols=+orbeta,+ci",
        "--pheno", covariate,
        "--mpheno", "5",
        "--max-maf", "0.05",
        "--mac", "2",
        "--adjust",
        "--allow-no-sex",
        "--ci", "0.95",
        "--covar", covariate,
        "--covar-variance-standardize",
        "--covar-name", "SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
    ]
  
    # Run for each var_class
    if var_class:
        variantsToKeep = f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.{var_class}.variantstoKeep.txt"
        prefix_variant = f"{prefix}_{var_class}"
    
        plinkAssoc = plinkAssoc_base + ["--extract", "range", variantsToKeep, "--out", prefix_variant]
        subprocess.run(plinkAssoc, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
        plinkGLM = plinkGLM_base + ["--extract", "range", variantsToKeep, "--out", prefix_variant]
        subprocess.run(plinkGLM, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    else:
        # Run with --out prefix_base_all
        plinkAssoc = plinkAssoc_base + ["--out", f"{prefix}_all"]
        subprocess.run(plinkAssoc, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
        plinkGLM = plinkGLM_base + ["--out", f"{prefix}_all"]
        subprocess.run(plinkGLM, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)


In [ ]:
for ances in ANCESTRIES:
    for classes in  variant_classes:
        variantAssoc("CADPS2", ances, classes)

In [ ]:
for var_class in variant_classes:
        for ANCESTRY in ANCESTRIES:
            try:
                glm_file = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_CADPS2_{var_class}.PHENO.glm.logistic.hybrid"
                df = pd.read_csv(glm_file, sep='\t') 
                df = df.rename(columns={'LOG(OR)_SE': 'SE'})
                df.to_csv(glm_file, sep='\t', index=False)
            except:
                pass

## Summary

In [ ]:
import shutil
import glob

classes1 = ('allVariants','allExonic')
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for i in ANCESTRIES:
    current_dir = os.path.join(ANA_DIR, str(i))
    summary_dir = os.path.join(current_dir, "Summary")
    if not os.path.exists(summary_dir):
        os.makedirs(summary_dir)

    pattern = os.path.join(current_dir, f"{i}_*")
    for file_path in glob.glob(pattern):
        if os.path.isfile(file_path):
            file_name = os.path.basename(file_path)
            destination = os.path.join(summary_dir, file_name)
            shutil.move(file_path, destination)

In [ ]:
for GENE_NAME in GENE_DICT:
    for ANCESTRY in ANCESTRIES:
        for class1 in classes1:
            try:
                print(f"Working on {ANCESTRY}, class: {class1}\n")
                ## Input files
                summary_dir = f"{ANA_DIR}/{ANCESTRY}/Summary"
                base_path = f'{summary_dir}/{ANCESTRY}_{GENE_NAME}'
                glm = f"{base_path}.glm.{class1}"
                assoc = f"{base_path}.assoc.{class1}"
                
                glm_df = pd.read_csv(f"{glm}.txt", sep='\t', header=0)
                glm_adj_df = pd.read_csv(f'{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_CADPS2.glm.{class1}.PHENO1.glm.logistic.hybrid.adjusted', sep='\t', header=0)            
                glm_merge = pd.merge(glm_df, glm_adj_df[['ID', 'BONF']], how='inner', on='ID')
                
                assoc_df = pd.read_csv(f"{assoc}.txt", sep='\t', header=0)   
                assoc_adj_df = pd.read_csv(f'{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_CADPS2.assoc.{class1}.assoc.adjusted', sep='\s+', header=0)           
                assoc_merge = pd.merge(assoc_df, assoc_adj_df[['SNP', 'BONF']], how='inner', on='SNP')
                assoc_merge['ID'] = assoc_merge['SNP']
                assoc_merge['A1_FREQ'] = assoc_merge['F_A']

                os.makedirs(summary_dir, exist_ok = True)

                glm_out = f"{summary_dir}/{ANCESTRY}_{GENE_NAME}_{class1}.glm"
                assoc_out = f"{summary_dir}/{ANCESTRY}_{GENE_NAME}_{class1}.assoc"
                glm_merge.to_csv(glm_out, index=False)
                assoc_merge.to_csv(assoc_out, index=False)
                
            except Exception as e:
                print(e)
            pass

In [ ]:
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for ANCESTRY in  ANCESTRIES:
    for classes in classes1:
        try:
            summary_dir = f"{ANA_DIR}/{ANCESTRY}/Summary"
            base = f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}"
            df_glm = pd.read_csv(f"{base}.glm", sep=',', header=0)
            df_assoc = pd.read_csv(f"{base}.assoc", sep=',', header=0) 

            df_glm.to_csv(f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}_processed_glm.csv", index=False)
            df_assoc.to_csv(f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}_processed_assoc.csv", index=False)              
        except:
            pass

summary_stats = []
analysis1 = {'glm', 'assoc'}
for ANCESTRY in ANCESTRIES:
    for classes in classes1:
        for a in analysis1:
            try:
                summary_dir = f"{ANA_DIR}/{ANCESTRY}/Summary"
                base = f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}_processed_{a}.csv"
                print(f"Read {base}")
                df = pd.read_csv(base, header=0)
                df.insert(0, "Gene", GENE_NAME)
                df.insert(1, "Ancestry", ANCESTRY)
                df.insert(2, "Analysis", a)
                df.insert(3, "Variant type", classes)
                summary_stats.append(df)
            except Exception as e:
                print(e)
                pass
                    
summary_stats = pd.concat(summary_stats, ignore_index=True) if summary_stats else pd.DataFrame()
summary_stats

In [ ]:
summary_stats.columns

In [ ]:
common_cols = ['Gene','Ancestry','Analysis','Variant type',	
               'ID', 'A1', 'A1_FREQ', 'P', 'BONF', 
               'OR', 'SE', 'L95', 'U95', 
               'Hom Cases', 'Het Cases', 'Hom Ref Cases', 
               'Missing Cases', 'Total Cases', 'Carrier Freq in Cases', 
               'Hom Controls', 'Het Controls', 'Hom Ref Controls', 
               'Missing Controls', 'Total Controls', 'Carrier Freq in Controls']

In [ ]:
summary_stats = summary_stats[common_cols]
summary_stats

In [ ]:
summary_stats['Variant type'].value_counts().to_frame()

In [ ]:
summary_stats.to_csv(f"{RESU_DIR}/{analysis}/{GENE_NAME}_{analysis}_assocglm_SummaryStats.csv",  index=False)

### Significant stats

In [ ]:
significant_stats = summary_stats[
    (summary_stats['P'] <= 0.05) &
    (summary_stats['Hom Cases'] != 0) &
    (summary_stats['Het Cases'] != 0) &
    (summary_stats['Hom Controls'] != 0) &
    (summary_stats['Het Controls'] != 0)
]
significant_stats

In [ ]:
gene = 'CADPS2'
significant_stats.to_csv(f"{RESU_DIR}/{analysis}/{gene}_{analysis}_assocglm_SignificantVariantsNonAdjusted.csv", index=False)

In [ ]:
significant_stats_bonf = significant_stats[(significant_stats['BONF'] <= 0.05)]
significant_stats_bonf

In [ ]:
significant_stats_bonf.to_csv(f"{RESU_DIR}/{analysis}/CADPS2_{analysis}_assocglm_SignificantVariantsAdjusted.csv", index=False)

## Annotate significant variants

In [ ]:
significant_stats[['chr', 'pos', 'ref', 'alt']] = significant_stats['ID'].str.split(':', expand=True)

# Prepare avinput DataFrame
avinput = significant_stats[['chr', 'pos', 'pos', 'ref', 'alt']]

# Save to tab-delimited file
avinput.to_csv(f'{RESU_DIR}/significant_variants.avinput', sep='\t', index=False, header=False)

In [ ]:
BUILD = "hg38"
annotateClinvar = [
        "perl", f"{TOOL_DIR}/annovar/table_annovar.pl",
        f'{RESU_DIR}/significant_variants.avinput',
        f"{TOOL_DIR}/annovar/humandb/",
        "-buildver", f"{BUILD}",
        "-out", f'{RESU_DIR}/{analysis}/significant_variants',
        "-remove", 
        "-protocol", "clinvar_20250721",
        "-operation", "f",
        "--nopolish",
        "-nastring", ".",
    ]

subprocess.run(annotateClinvar, check=True)

In [ ]:
multianno = pd.read_csv(f'{RESU_DIR}/{analysis}/significant_variants.hg38_multianno.txt', sep='\t', header=0)
multianno['ID'] = multianno['Chr'] + ":" + multianno['Start'].astype(str) + ":" + multianno['Ref'] + ":" + multianno['Alt']

annotated_significant = pd.merge(significant_stats, multianno, how='inner', on='ID')
annotated_significant

In [ ]:
significant_stats_bonf.to_csv(f"{RESU_DIR}/{analysis}/CADPS2_{analysis}_assocglm_SignificantVariantsAnnotated.csv", index=False)

## To-Do list

- Make all analyses into one single function, and call it with different assoc models, ANCESTRIES, variant groups in parallel

- Check exonic GLM +

- Add all outputs in single df by ANCESTRIES in first column, keep p<0.05